In [8]:
import os, yaml, sys
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import loadmat
import h5py, cv2, joblib
from torchvision.models.feature_extraction import create_feature_extractor
from IPython.display import clear_output
ENV = os.getenv("MY_ENV", "dev")
with open("../../config.yaml", "r") as f:
    config = yaml.safe_load(f)
paths = config[ENV]["paths"]
sys.path.append(paths["src_path"])
sys.path.append(paths["useful_stuff_path"])
from useful_stuff.general_utils.utils import TimeSeries, print_wise
from project_specific_utils.dataloader import load_eyetracking_data
from image_processing.utils import read_video
from image_processing.gaze_dep_models import sequential_gaze_dep_mod, save_ipca_patch

In [2]:
from dataclasses import dataclass, field

@dataclass
class Cfg:
    sub_num = 3
    fs = 50
    model_name = "pixelwise_lum";
    sq_side = 250
    downsampling_sq_size = 50
cfg = Cfg()

In [ ]:
def ANN_extraction_projection_1917_wrapper(paths: dict[str: str], rank: int, sub_num: int, model, sq_side: int, input_size, model_name: str, layer_name, n_components, pooling, fs, device, screen_res=(1080, 1920), secs_to_skip=5,): 
    ipca_path = save_ipca_patch(paths, model_name, layer_name, n_components, sq_side, pooling,) # model_name, layer_name, n_components, sq_size, pooling
    ipca_obj = joblib.load(ipca_path)
    PCs = ipca_obj.components_.T
    print_wise(f"Start running {model_name} for sub {sub_num}", rank=rank)
    for irun in range(1, 7):
        gaze_dep_ANN_extraction(paths, rank, sub_num, sq_side, model, model_name, layer_name, n_components, pooling, PCs, input_size, irun, fs, device, screen_res=(1080, 1920), secs_to_skip=5, )
    # end for irun in range(1, 7):
# EOF


In [ ]:
def save_ANN_features(paths, full_model_name, fs, sub_num, run, n_components, sq_side, pooling,):
    save_name = f"{paths['data_path']}/models/sub{sub_num:03d}_run{run:02d}_{full_model_name}_{n_components}components_{pooling}pooling_gazedep_{sq_side}x{sq_side}_{round(fs)}Hz.h5"
    return save_name
# EOF

# TODO run function to see if it's working (listen to audio)

In [ ]:
from project_specific_utils.utils import run2part
from image_processing.gaze_dep_models import pad_frame, extract_square_patch, extract_features_1917_movie
from useful_stuff.image_processing.utils import get_video_dimensions, preprocess_batch
import torch
def gaze_dep_ANN_extraction(paths: dict[str: str], rank: int, sub_num: int, sq_side: int, model, model_name: str,layer_name, n_components, pooling, PCs, input_size,  run: int, fs, device, screen_res=(1080, 1920), secs_to_skip=5, **kwargs): 
    movie_part = run2part(run)
    movie_fn = f"{paths['data_path']}/stimuli/Project1917_movie_part{movie_part}_24Hz.mp4"
    cap = cv2.VideoCapture(movie_fn)
    fps = cap.get(cv2.CAP_PROP_FPS)
    cap.set(cv2.CAP_PROP_POS_FRAMES, round(5*fps))
    h, w, frames_n = get_video_dimensions(cap)
    save_name = save_ANN_features(paths, f"{model_name}_{layer_name}", fs, sub_num, run, n_components, sq_side, pooling,)
    if os.path.exists(save_name):
        print_wise(f"model already exists at {save_name}", rank=rank)
        return None
    # end if os.path.exists(save_name):
    feature_extractor = create_feature_extractor(
        model, return_nodes=[layer_name]
    ).to(device)
    xy_gaze, _ = load_eyetracking_data(paths, sub_num, run, fs, xy=True)
    xy_gaze.resample(fps)
    frames_n -= round(secs_to_skip*fps) +2 # to be on the safe side, because when downsampled, the number of gaze-datapoints exceeds the number of frames
    if frames_n > len(xy_gaze):
        raise IndexError(f"The number of frames ({frames_n}) is larger than the number of gaze datapoints ({len(xy_gaze)}) in sub {sub_num} run {run}")
    # end if frames_n > len(xy_gaze):

    offset_dims = ((screen_res[0] -h)//2 , ( screen_res[1] - w)//2)
    canvas = None
    features = []
    for frame_idx in range(frames_n):
        xy = xy_gaze[frame_idx]
        ret, frame = cap.read()
        if not ret:
            raise RuntimeError(f"Failed to read frame {frame_idx} from {movie_fn}")
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        canvas = pad_frame(frame, (h,w), offset_dims,)
        frame_patch = extract_square_patch(canvas, round(xy[0]), round(xy[1]), sq_side)
        frame_patch = torch.from_numpy(frame_patch)
        frame_patch = preprocess_batch(frame_patch[None, :, :, :], input_size, device=device)
        curr_features = extract_features_1917_movie(frame_patch, feature_extractor, layer_name, input_size, pooling=pooling, device=device)
        curr_features = curr_features @ PCs # TODO check if features are a column vector
        features.append(curr_features)
        # canvas = sequential_gaze_dep_loop(cap, xy_gaze, frame_idx, sq_side, (h,w), offset_dims, canvas, features, func, rank, sub_num, run, *args, **kwargs)
    # end for frame_idx in range(frames_n):
    
    features = np.stack(features, axis=1)
    with h5py.File(save_name, "w") as f:
        f.create_dataset("vecrep", data=features)
    # end with h5py.File(save_name, "w") as f:
    print_wise(f"model {model_name} saved at {save_name}", rank=rank)
# EOF

